# 14. MLflow Tracing, Prompt Versioning, and Evaluation for the Agentic Chatbot

**RAG Pipeline Series — Notebook 14**

[Notebook 13](13_autonomous_agentic_rag_langgraph.ipynb) built a LangGraph agent that can search `rag.pdf`, search the live web, and remember earlier turns in a conversation. It works — but "it works" is not something you can measure, compare, or debug from the outside. If you tweak the system prompt, did the agent actually get better, or does it just feel that way after two manual tries? If a user reports a bad answer, what did the agent actually see and do on that turn?

This notebook wires the same agent from notebook 13 into **[MLflow](https://mlflow.org/docs/latest/genai/)**'s GenAI tooling to answer both questions, without changing the agent's behavior at all:

- **Prompt versioning** — the hard-coded `AGENT_SYSTEM_PROMPT` string from notebook 13 becomes a versioned prompt in MLflow's Prompt Registry (`mlflow.genai.register_prompt`), so every change is a new version with a diffable history, and the running agent always loads whichever version an alias (e.g. `champion`) currently points to.
- **Autologged tracing** — one line, `mlflow.langchain.autolog()`, turns every graph invocation into a full trace: the LLM call, every tool call (`search_rag_document`, `web_search`), and their inputs/outputs, all nested under one span per turn and viewable in the MLflow UI.
- **LLM-as-judge evaluation** — `mlflow.genai.evaluate()` runs the agent over a small labeled question set and scores each answer with LLM-judge scorers (MLflow's built-in `Correctness`/`Guidelines`/`RelevanceToQuery`, plus one fully custom judge), turning "does this feel right" into a number you can track across prompt versions.

**This notebook assumes a local MLflow tracking server is already running** at `http://localhost:5000` (e.g. `mlflow server --host 127.0.0.1 --port 5000` in a separate terminal) — start that first. Because it depends on a *local* server reachable at `localhost`, this notebook is written to run locally rather than in Colab (unlike notebooks 1-13).

## Setup

In [ ]:
%pip install -q -U langchain langchain-community langchain-core langchain-text-splitters sentence-transformers langchain-huggingface langchain-chroma chromadb langchain-google-genai langgraph python-dotenv gradio ddgs "mlflow>=2.20.0"

### Files this notebook needs

- `rag.pdf` — the source document (same as every other notebook in the series), resolved via `resolve_pdf_path()`.
- `.env` — must contain a `GOOGLE_API_KEY` for Gemini. Get a free key from [Google AI Studio](https://aistudio.google.com/apikey).
- A running MLflow tracking server at `http://localhost:5000` (see the note above) — everything this notebook logs (prompts, traces, evaluation runs) goes there.

No key is needed for web search — `ddgs` queries DuckDuckGo directly.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_path = "/content/.env" if os.path.exists("/content/.env") else ".env"
assert os.path.exists(env_path), "Could not find .env - create one with GOOGLE_API_KEY first."
load_dotenv(env_path)

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set - check your .env file."
print("GOOGLE_API_KEY loaded.")

GOOGLE_API_KEY loaded.


## 1. Connect to the MLflow tracking server

`mlflow.set_tracking_uri()` points every MLflow call in this notebook - prompt registry writes, autologged traces, evaluation runs - at the local server instead of the default `./mlruns` folder. `mlflow.set_experiment()` groups everything this notebook logs under one named experiment, so it's easy to find in the UI later.

In [3]:
%pip install -q -U mlflow

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
import mlflow

MLFLOW_TRACKING_URI = "http://localhost:5000"  # assumes `mlflow server` is already running locally
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("rag-notebooks-agentic-chatbot")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/14 22:01:24 INFO mlflow.tracking.fluent: Experiment with name 'rag-notebooks-agentic-chatbot' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


## 2. Rebuild the retriever

Identical to notebooks 7-13: load `rag.pdf`, strip headers/footers, chunk chapter-by-chapter, embed with the series' `sentence-transformers/paraphrase-MiniLM-L3-v2` model, and index into an in-memory Chroma store.

In [3]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks

pages, full_text, chapters, chunks = load_chapter_chunks()

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"{len(chunks)} chunks indexed; retriever returns top 5 matches per query")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 14856.18it/s]


181 chunks indexed; retriever returns top 5 matches per query


## 3. Tools

Identical to notebooks 12-13: `search_rag_document` wraps the `retriever` built above, `web_search` wraps a key-less DuckDuckGo text search via `ddgs`.

In [4]:
from langchain_core.tools import tool

tool_call_log = []  # cleared per turn in run_agent_turn(); UI/eval code reads this for transparency


def format_context(docs):
    return "\n\n".join(
        f"[Chapter {d.metadata['chapter_num']}: {d.metadata['chapter_title']}]\n{d.page_content}"
        for d in docs
    )


@tool
def search_rag_document(query: str) -> str:
    """Search rag.pdf, the RAG course document, for relevant passages.

    Use this for any question about RAG concepts covered in the course: chunking,
    embeddings, vector stores, keyword/dense/hybrid retrieval, re-ranking,
    augmentation, or generation. Always try this tool first for conceptual
    questions about RAG - it is the authoritative source for the course material.
    """
    docs = retriever.invoke(query)
    result = format_context(docs)
    tool_call_log.append({
        "tool": "search_rag_document",
        "query": query,
        "summary": ", ".join(f"Chapter {d.metadata['chapter_num']}" for d in docs),
    })
    return result


@tool
def web_search(query: str) -> str:
    """Search the live web for current information that rag.pdf would not contain.

    Use this for questions about recent events, current facts, or any topic outside
    the RAG course document - anything time-sensitive or not related to RAG concepts.
    """
    from ddgs import DDGS

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))

    tool_call_log.append({
        "tool": "web_search",
        "query": query,
        "summary": ", ".join(r["title"] for r in results) if results else "no results",
    })

    if not results:
        return "No web results found."
    return "\n\n".join(f"{r['title']}\n{r['body']}\nSource: {r['href']}" for r in results)


TOOLS = [search_rag_document, web_search]

## 4. Prompt versioning with MLflow's Prompt Registry

Notebook 13 hard-coded `AGENT_SYSTEM_PROMPT` as a Python string. That makes every edit an untracked code change - no history, no way to compare "the prompt the agent used last Tuesday" against today's. `mlflow.genai.register_prompt()` fixes that: each call creates a new, immutable version of a named prompt, and an **alias** (e.g. `champion`) points at whichever version should actually be used right now - so promoting a new prompt is a metadata update, not a code change.

First, register the prompt from notebook 13 verbatim as version 1.

In [5]:
PROMPT_NAME = "rag-agent-system-prompt"

SYSTEM_PROMPT_V1 = (
    "You are a helpful assistant that answers questions using tools.\n"
    "- For questions about RAG concepts (chunking, retrieval, embeddings, re-ranking, "
    "augmentation, generation), call search_rag_document first.\n"
    "- For questions about current events, facts, or anything outside the RAG course "
    "document, call web_search.\n"
    "- If search_rag_document doesn't contain the answer, try web_search before giving up.\n"
    "- Answer only from tool results - do not rely on outside knowledge you were not given "
    "by a tool. Mention which tool(s) and, if applicable, which chapter(s) your answer came from.\n"
    "- You may be shown earlier turns from this same conversation - use them for context on "
    "follow-up questions (e.g. \"the second one\", \"what about that chapter\")."
)

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=SYSTEM_PROMPT_V1,
    commit_message="Initial system prompt, ported as-is from notebook 12/13's hard-coded AGENT_SYSTEM_PROMPT.",
)
print(f"Registered {PROMPT_NAME} v{prompt_v1.version}")

2026/07/14 22:12:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: rag-agent-system-prompt, version 1


Registered rag-agent-system-prompt v1


Now register a revised version that tightens the one instruction the LLM-as-judge scorer in section 8 will actually grade: instead of *optionally* mentioning a chapter, the agent must *name it explicitly* whenever it uses `search_rag_document`. It also caps answer length. Registering this as a new version - rather than editing `SYSTEM_PROMPT_V1` in place - keeps both versions permanently in the registry, so section 8's evaluation results can later be compared version-to-version instead of only ever seeing "the current prompt."

Setting the `champion` alias to point at v2 is what makes the agent below actually use it - the code that loads the prompt never hard-codes a version number, only the alias.

In [6]:
SYSTEM_PROMPT_V2 = (
    "You are a helpful assistant that answers questions using tools.\n"
    "- For questions about RAG concepts (chunking, retrieval, embeddings, re-ranking, "
    "augmentation, generation), call search_rag_document first.\n"
    "- For questions about current events, facts, or anything outside the RAG course "
    "document, call web_search.\n"
    "- If search_rag_document doesn't contain the answer, try web_search before giving up.\n"
    "- Answer only from tool results - do not rely on outside knowledge you were not given "
    "by a tool. When you use search_rag_document, you must explicitly name the chapter number "
    "and title your answer came from (e.g. 'Chapter 7: Retrieval Evaluation') - never just say "
    "'the document'.\n"
    "- Keep answers to 3-6 sentences unless the user explicitly asks for more detail.\n"
    "- You may be shown earlier turns from this same conversation - use them for context on "
    "follow-up questions (e.g. \"the second one\", \"what about that chapter\")."
)

prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=SYSTEM_PROMPT_V2,
    commit_message="Require an explicit chapter citation instead of an optional one; cap answer length.",
)

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=prompt_v2.version)
print(f"Registered {PROMPT_NAME} v{prompt_v2.version}; 'champion' alias now points at it.")

2026/07/14 22:12:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: rag-agent-system-prompt, version 2


Registered rag-agent-system-prompt v2; 'champion' alias now points at it.


In [7]:
# Load whatever version the "champion" alias currently points to - not a hard-coded version number.
active_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")
print(f"Loaded {PROMPT_NAME} v{active_prompt.version} (alias: champion)")
print(active_prompt.template)

Loaded rag-agent-system-prompt v2 (alias: champion)
You are a helpful assistant that answers questions using tools.
- For questions about RAG concepts (chunking, retrieval, embeddings, re-ranking, augmentation, generation), call search_rag_document first.
- For questions about current events, facts, or anything outside the RAG course document, call web_search.
- If search_rag_document doesn't contain the answer, try web_search before giving up.
- Answer only from tool results - do not rely on outside knowledge you were not given by a tool. When you use search_rag_document, you must explicitly name the chapter number and title your answer came from (e.g. 'Chapter 7: Retrieval Evaluation') - never just say 'the document'.
- Keep answers to 3-6 sentences unless the user explicitly asks for more detail.
- You may be shown earlier turns from this same conversation - use them for context on follow-up questions (e.g. "the second one", "what about that chapter").


## 5. Binding tools to Gemini

Same as notebooks 12-13: `llm.bind_tools(TOOLS)` returns a runnable that may respond with `tool_calls` instead of (or alongside) plain text.

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.2)
llm_with_tools = llm.bind_tools(TOOLS)

## 6. The graph

Unchanged from notebook 13: `agent`/`tools` nodes, a conditional edge (`tools_condition`), and a `MemorySaver` checkpointer for per-thread conversation memory. See notebook 13 for the full walkthrough of why this graph shape replaces a hand-rolled loop.

In [9]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition


def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # -> "tools" if tool_calls, else END
builder.add_edge("tools", "agent")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

## 7. Autologged tracing

`mlflow.langchain.autolog()` instruments LangChain and LangGraph so that every `graph.invoke()` call below automatically produces a trace in the experiment set in section 1 - no changes needed to the graph, the tools, or the LLM. Each trace nests the full call tree for that turn: the `agent` node's LLM call (prompt, response, token counts, latency), any `tools` node executions (`search_rag_document`/`web_search`, with their exact inputs and outputs), and the loop back to `agent` if the model chained multiple tool calls.

This is what turns "the agent answered wrong" from a mystery into something inspectable: open the MLflow UI's **Traces** tab for this experiment and see exactly what the model was given and what it decided to do, turn by turn.

In [10]:
mlflow.langchain.autolog(log_traces=True)

## 8. The agent loop: `run_agent_turn()`

Same shape as notebook 13's `run_agent_turn()` - a `SystemMessage` is only injected the first time a `thread_id` is seen, since the `MemorySaver` checkpointer already carries everything before it on later turns. Two things are new:

1. The system message now comes from `active_prompt.template` (the registry's `champion` version) instead of a hard-coded string.
2. `@mlflow.trace` wraps the whole function as one named, top-level span, and `mlflow.update_current_trace()` attaches `thread_id` and `prompt_version` as trace tags - so in the MLflow UI you can filter traces by conversation, or by which prompt version produced them, without opening each one.

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage


@mlflow.trace(name="run_agent_turn", span_type="AGENT")
def run_agent_turn(question, thread_id, max_steps=8):
    tool_call_log.clear()
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 2 * max_steps + 1}

    is_new_thread = not graph.get_state(config).values
    input_messages = [HumanMessage(content=question)]
    if is_new_thread:
        input_messages = [SystemMessage(content=active_prompt.template)] + input_messages

    mlflow.update_current_trace(
        tags={"thread_id": thread_id, "prompt_version": str(active_prompt.version)},
    )

    result = graph.invoke({"messages": input_messages}, config)
    return result["messages"][-1].content, list(tool_call_log)

Run a couple of turns to generate traces - one follow-up question on the same thread (to exercise memory), one on a fresh thread (to confirm isolation). After running these, check the MLflow UI's **Traces** tab: each call below should show up as its own trace, with the LLM call and any tool calls nested underneath.

In [12]:
demo_thread = "demo-thread-1"

answer, log = run_agent_turn("What is re-ranking used for in a RAG pipeline?", demo_thread)
print("ANSWER:\n", answer)
print("\nTOOL CALLS:")
for entry in log:
    print(f"  - {entry['tool']}({entry['query']!r}) -> {entry['summary']}")

ANSWER:
 [{'type': 'text', 'text': 'According to **Chapter 08: Re-ranking**, re-ranking is the second stage of a multi-stage retrieval funnel designed to improve precision. While the first stage focuses on high recall by quickly retrieving a large pool of candidates (e.g., top-100), the re-ranking stage uses a more powerful cross-encoder model to surgically select the most relevant documents (e.g., top-5). This funnel approach allows the system to achieve the best of both worlds: a fast initial search that casts a wide net and an expensive second stage that ensures high accuracy. Ultimately, re-ranking helps provide the most relevant context to the generator within an acceptable latency budget.', 'extras': {'signature': 'Eu0HCuoHARFNMg9DfDMTkyL5DT/Zm0jR39Wr+rH3c9pCCAbzv/WkV/E1+O9br18oWY5hGeliISeRbgvR7FLp4o5Nh60YOL5rqW8lpS0xWZ3jkR3piom7SkjuQz7Jk8gcQujDzRCVKNp85M1TwMssaCgPiPFwCpj1kqqrxpHVa6mBx0M0DEtfCstMa69sWRI+T7MbFH1/LFKVI5daqNJwMSzyxJEH3iVYa8lC+/CYhMsXjx9wKW+yAG1A07Y8Qgkrgf+DxxPMuxSyz

In [13]:
answer, log = run_agent_turn("Which chapter did that come from, and what's the very next chapter about?", demo_thread)
print("ANSWER:\n", answer)
print("\nTOOL CALLS:")
for entry in log:
    print(f"  - {entry['tool']}({entry['query']!r}) -> {entry['summary']}")

ANSWER:
 [{'type': 'text', 'text': 'The information about re-ranking came from **Chapter 08: Re-ranking**. The very next chapter is **Chapter 09: Augmentation**, which focuses on how to format, label, and order retrieved chunks for the LLM. This chapter explains that good augmentation involves selecting, structuring, and framing information to ensure the model produces accurate and grounded responses.'}]

TOOL CALLS:
  - search_rag_document('Chapter 09 title') -> Chapter 03, Chapter 03, Chapter 09, Chapter 03, Chapter 09


Trace(trace_id=tr-c3008b35ac1bafc43061dbc76ba887c2)

In [14]:
answer, log = run_agent_turn("Who won the most recent Super Bowl?", "demo-thread-2")
print("ANSWER:\n", answer)
print("\nTOOL CALLS:")
for entry in log:
    print(f"  - {entry['tool']}({entry['query']!r}) -> {entry['summary']}")

ANSWER:
 [{'type': 'text', 'text': "The Kansas City Chiefs won the most recent Super Bowl, Super Bowl LVIII, which took place on February 11, 2024. They defeated the San Francisco 49ers with a final score of 25-22 in an overtime thriller. This victory marked the Chiefs' second consecutive title, making them the first team to win back-to-back Super Bowls in 20 years. Quarterback Patrick Mahomes was named the Super Bowl MVP after throwing the game-winning touchdown pass.", 'extras': {'signature': 'Ev8CCvwCARFNMg9aT1Ej4jLs3rZsmeVKCFuuUULRxgXhe6hv6q46m8QVwlXnfezhJHj3ZZDxwzcYRN7Avbh2Jy1xm8C1Ry5X0wA/8yKpjwzh34cUvteP1xhRP1CuGHFUELZp5nCWLIMps8CWaSinZOcM3LGePxzcKDHHIUysv8fMfap+Mv8DKchbWxqcISJ+SwvKKYDTqvA0IjByrmpQoSoInwFTi+9DIDilIJam7U5pE0O6dVbrbhkEIRzu8CNB5Te4QqY17MspziRtUiSDIU9MCKI3Y6cLx47OBVmPOYccwBUuQoctRXmIRm2XouDKoZ4vQurQAYUqQnwYk5zs877uEeXJ91f5b9ze1EO9TyPnjYc6dgM2inRVrtlcil7p4pV5d0B8CmFGNHGnSd9Q6I8BKceWbjDHnr5C/y/s1Eb1Mc4KIePyriK5zYjVD4wKr5xuYwqbyQBoJgE+mDYhLE32n8bbYHFOjjaMgyArp7CGJ+DLBPl

[Trace(trace_id=tr-205eb6a483f077ec56c4b11a700a0e9c), Trace(trace_id=tr-38c4c50cc2474809721640fa447a6604)]

## 9. Evaluating the agent with LLM-as-judge scorers

Traces answer "what did the agent do on this one turn"; evaluation answers "how good is the agent, on average, over many turns". `mlflow.genai.evaluate()` runs `run_agent_turn()` over a labeled set of questions and scores every answer with one or more **scorers** - most of them themselves LLM calls ("LLM-as-judge") that grade the answer against the question, against ground truth, or against a written guideline.

The eval set below reuses `rag_utils.EVAL_QUERIES` - the same 8 questions (one per chapter of `rag.pdf`) that notebooks 9 and 10 use to score *retrieval* quality. Here they double as ground truth for scoring the *agent's final answer*: each question is paired with a short substring known to appear in its correct source chunk, used as an `expected_facts` expectation.

In [21]:
import uuid

from rag_utils import EVAL_QUERIES

eval_data = [
    {
        "inputs": {"question": question},
        "expectations": {"expected_facts": [expected_substring]},
    }
    for question, expected_substring in EVAL_QUERIES
]


def predict_fn(question):
    # Each eval row gets its own thread so evaluation runs never bleed into each other's memory.
    thread_id = f"eval-{uuid.uuid4()}"
    answer, _ = run_agent_turn(question, thread_id)
    return answer

### A custom LLM-as-judge scorer

MLflow ships several built-in judges (`Correctness`, `Guidelines`, `RelevanceToQuery`, ...) that cover the common cases below. But section 4's whole point was making chapter citation a *required, checkable* behavior - so this notebook also defines one fully custom judge with `mlflow.genai.judges.custom_prompt_judge()`: an LLM call whose only job is to read the question and answer and decide whether a chapter was actually named. `{{inputs}}`/`{{outputs}}` in the template are filled in from each eval row's `inputs`/the agent's returned answer.

In [22]:
from mlflow.genai import scorer
from mlflow.genai.judges import custom_prompt_judge

_cites_source_chapter_judge = custom_prompt_judge(
    name="cites_source_chapter",
    prompt_template=(
        "You are grading a RAG chatbot's answer for whether it cites its source.\n\n"
        "Question: {{inputs}}\n"
        "Answer: {{outputs}}\n\n"
        "Does the answer explicitly name a chapter of the course document it drew information "
        "from (e.g. 'Chapter 7' or 'Chapter 7: Retrieval Evaluation')? Saying only 'the document' "
        "or 'the course material' does not count.\n\n"
        "You must choose one of the following categories.\n\n"
        "[[yes]]: The answer explicitly names a chapter of the course document.\n"
        "[[no]]: The answer does not name a specific chapter."
    ),
    numeric_values={"yes": 1.0, "no": 0.0},
)


# `custom_prompt_judge()` returns a plain callable, not a `Scorer` - `mlflow.genai.evaluate()`
# only accepts `Scorer` instances, so `@scorer` wraps it into one. The wrapper's parameter names
# (`inputs`, `outputs`) tell MLflow what to feed in from each eval row / predict_fn call.
@scorer(name="cites_source_chapter")
def citation_judge(inputs, outputs):
    return _cites_source_chapter_judge(inputs=inputs, outputs=outputs)

C:\Users\shrin\AppData\Local\Temp\ipykernel_12500\1999017566.py:4: FutureWarning: ``mlflow.genai.judges.custom_prompt_judge.custom_prompt_judge`` is deprecated since 3.4.0. This method will be removed in a future release. Use ``mlflow.genai.make_judge`` instead.
  _cites_source_chapter_judge = custom_prompt_judge(


### Run the evaluation

`Correctness` checks the answer against `expected_facts`; `Guidelines` checks it against a written rule in plain English; `RelevanceToQuery` checks the answer actually addresses what was asked; `citation_judge` is the custom scorer defined above. All four are themselves LLM calls, graded and logged as part of the same MLflow run.

In [23]:
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        Correctness(),
        Guidelines(
            name="cites_chapter",
            guidelines="The response must mention which chapter of the RAG course document the answer came from.",
        ),
        RelevanceToQuery(),
        citation_judge,
    ],
)

2026/07/14 22:32:22 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/07/14 22:32:25 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/07/14 22:32:25 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
Evaluating: 100%|██████████| 8/8 [Elapsed: 01:30, Remaining: 00:00] [predict_fn: 62%, scorers: 38%]


### Inspect the results

`results.metrics` has the per-scorer aggregate (e.g. mean `Correctness`, mean `cites_source_chapter`); `results.tables["eval_results"]` has the per-question breakdown, including each scorer's written rationale for its score. The same run is also logged to the MLflow server, under the experiment set in section 1 - the **Evaluations** tab gives a UI for the same table, and lets a future run (e.g. after promoting a v3 prompt) be compared side-by-side against this one.

In [24]:
print(results.metrics)
results.tables["eval_results"]

{'cites_chapter/mean': np.float64(1.0), 'correctness/mean': np.float64(0.5), 'relevance_to_query/mean': np.float64(1.0), 'cites_source_chapter/mean': np.float64(1.0)}


,trace_id,cites_chapter/value,cites_chapter/rationale,relevance_to_query/value,relevance_to_query/rationale,cites_source_chapter/value,cites_source_chapter/rationale,correctness/value,correctness/rationale,expected_facts/value,...,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-10ade6badb3002a60759b6703c4811e7,yes,The guideline requires that the response must ...,yes,The question asks about what the Okapi BM25 fo...,1.0,The answer includes the phrase 'According to C...,no,The claim simply states '- Okapi Best Match 25...,None,...,None,OK,1784048559918,35575,{'question': 'What does the Okapi BM25 formula...,"[[{'type': 'text', 'text': 'According to Chapt...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'EK3mutswAqYHWbZwPEgR5w==', 'spa...",[{'assessment_id': 'a-bee8c2e6ed3942d18c34885e...
1,tr-e5fbcefaf21c777f0e8a83529a554461,yes,The only provided guideline is that the respon...,yes,The question asks how giving a language model ...,1.0,The question is whether the answer explicitly ...,no,"The claim states '- dynamic, external knowledg...",None,...,None,OK,1784048559922,37859,{'question': 'How can giving a language model ...,"[[{'type': 'text', 'text': 'According to **Cha...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '5fvO+vIcd38OioNSmlVEYQ==', 'spa...",[{'assessment_id': 'a-26194fc13d26419e8ed6345e...
2,tr-03ade99de903e18d1ee1b5602e214ad3,yes,The guideline states that the response must me...,yes,The question asks about what kind of queries k...,1.0,The answer explicitly states 'According to Cha...,no,The question asks about what kind of queries k...,None,...,None,OK,1784048559924,42943,{'question': 'What kind of queries is keyword ...,"[[{'type': 'text', 'text': 'According to Chapt...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'A63pnekD4Y0e4bVgLiFK0w==', 'spa...",[{'assessment_id': 'a-3bb8754e2c404310935da7be...
3,tr-a0535f1731c5e12cc0558b17bbc04f90,yes,The single guideline provided states that the ...,yes,The question asks why reciprocal rank fusion i...,1.0,The answer explicitly states information is 'A...,yes,The claim states that RRF is parameter-light a...,None,...,None,OK,1784048559926,31963,{'question': 'Why is reciprocal rank fusion co...,"[[{'type': 'text', 'text': 'According to Chapt...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'oFNfFzHF4SzAVYsXu8BPkA==', 'spa...",[{'assessment_id': 'a-21f500afa2324924b36c4c47...
4,tr-bf1e4b3e475f6c9159391700f42da455,yes,The guideline states that the response must me...,yes,The question asks which retrieval metric matte...,1.0,The answer explicitly states 'According to Cha...,yes,Let's think step by step: The claim states tha...,None,...,None,OK,1784048559927,52040,{'question': 'Which retrieval metric matters m...,"[[{'type': 'text', 'text': 'According to **Cha...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'vx5LPkdfbJFZORcA9C2kVQ==', 'spa...",[{'assessment_id': 'a-98156dd02b4044429ab454c7...
5,tr-c08d7732c5d7f36723932c1d2bfdd118,yes,The guideline requires that the response must ...,yes,The question asks why a bi-encoder cannot capt...,1.0,The question asks whether the answer explicitl...,yes,The claim states that a bi-encoder cannot cons...,None,...,None,OK,1784048559927,32943,{'question': 'Why can't a bi-encoder capture f...,"[[{'type': 'text', 'text': 'According to Chapt...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'wI13MsXX82cjkywdK/3RGA==', 'spa...",[{'assessment_id': 'a-4e78b0404c114f409bb88f31...
6,tr-b3c728fe7286545b38001dc477df17cb,yes,The guideline requires that the res

## Takeaways

- **Prompt versioning** turns a system prompt from an untracked string literal into a registered, versioned artifact (`mlflow.genai.register_prompt`) that the running agent loads by **alias** (`champion`) rather than by hard-coded version - so promoting a new prompt is a one-line alias update, not a code change, and every past version stays inspectable.
- **`mlflow.langchain.autolog()`** instruments the exact same LangGraph agent from notebook 13 with zero changes to its logic - every turn becomes a trace showing the LLM call and every tool call it made, nested and timestamped, in the MLflow UI.
- **`mlflow.genai.evaluate()`** replaces "I tried a few questions and it seemed fine" with a scored, repeatable run over a labeled question set - using both MLflow's built-in LLM judges (`Correctness`, `Guidelines`, `RelevanceToQuery`) and a fully custom one (`custom_prompt_judge`) written specifically to check the behavior the v2 prompt was written to enforce.
- None of this changed what the agent *does* - notebook 13's graph, tools, and control flow are untouched. What changed is whether you can see what it did, and prove whether a change to it helped.

Together, prompt versioning + tracing + evaluation are what separate "an agent that works on my machine" from "an agent whose behavior is measured and its prompt changes gated on that measurement" - the same MLflow GenAI surface (`mlflow.genai.register_prompt`/`load_prompt`, `mlflow.langchain.autolog`, `mlflow.genai.evaluate`, `@mlflow.trace`) generalizes directly to any other LLM application, agentic or not.